# Exercise 2: Save Deduplicated Results to Apache Iceberg

## Learning Objectives

In this exercise, you will:
- Load either Exercise 1 Parquet **or** the project local CSV
- Optionally write a **local** Iceberg table (lab / session only)
- Publish a table into the **shared CDW Hive warehouse** so it appears in **Hue**
- Verify via Spark SQL and/or the Hive JDBC connection

## Prerequisites

1. Run **`00_Getting_Started.ipynb`**. For the Parquet path, also run **`01_Basic_Deduplication.ipynb`**.
2. Input options (set `INPUT_SOURCE` in Step 1):

| `INPUT_SOURCE` | File | Table name (default) |
|----------------|------|----------------------|
| `exercise1_parquet` | `/tmp/.../exercise1_exact.parquet` | `deduped_customers` |
| `local_csv` | `../data/redundant_data.csv` | `raw_customers` |

3. For **Hue-visible** tables: use the Hive Virtual Warehouse Data Connection (e.g. `go01-obsr-de`) whose JDBC looks like:

```text
jdbc:hive2://hs2-default-hive-aws.dw-go01-demo-aws.ylcu-atmi.cloudera.site/default;transportMode=http;httpPath=cliservice;ssl=true;...
```

4. Local `/tmp` Iceberg warehouses are **not** visible in Hue — only tables registered in the shared Hive Metastore / CDW.


## Step 1: Configure Connection and Input Source

Set the CAI Data Connection name (preferred). Choose `INPUT_SOURCE` to load Exercise 1 Parquet or the project local CSV (writes a differently named Iceberg table). Optional REST catalog values are used only when CML Spark is unavailable.


In [ ]:
import os
from pathlib import Path

# --- Preferred: CAI Project Data Connection (Spark Data Lake / Iceberg) ---
CONNECTION_NAME = os.environ.get("CML_CONNECTION_NAME", "go01-obsr-de")

# --- Input source: "exercise1_parquet" (default) or "local_csv" ---
INPUT_SOURCE = os.environ.get("ICEBERG_INPUT_SOURCE", "exercise1_parquet").strip().lower()
# Flip to local CSV without env vars:
# INPUT_SOURCE = "local_csv"

NAMESPACE = os.environ.get("ICEBERG_NAMESPACE", "cdp_user_demo")

# Distinct Iceberg table names per source
TABLE_NAME_DEDUPED = os.environ.get("ICEBERG_TABLE_DEDUPED", "deduped_customers")
TABLE_NAME_RAW = os.environ.get("ICEBERG_TABLE_RAW", "raw_customers")

def _resolve_local_csv() -> Path:
    env = os.environ.get("LOCAL_CSV")
    if env:
        return Path(env).resolve()
    candidates = [
        Path("../data/redundant_data.csv"),
        Path("data/redundant_data.csv"),
        Path.cwd() / "data" / "redundant_data.csv",
        Path.cwd().parent / "data" / "redundant_data.csv",
        Path.cwd() / "use-case-phase-1" / "data" / "redundant_data.csv",
    ]
    for p in candidates:
        if p.is_file():
            return p.resolve()
    return Path("../data/redundant_data.csv").resolve()


LOCAL_CSV = _resolve_local_csv()
LOCAL_PARQUET = Path(
    os.environ.get(
        "EXERCISE1_PARQUET",
        "/tmp/cdp_user_demo/phase1/results/exercise1_exact.parquet",
    )
)

# Auto-fallback: Exercise 1 output lives under /tmp and disappears after session restart
if INPUT_SOURCE in ("exercise1_parquet", "parquet") and not LOCAL_PARQUET.exists():
    if LOCAL_CSV.exists():
        print(
            f"⚠ {LOCAL_PARQUET} not found (re-run notebook 01, or it was cleared with /tmp).\n"
            f"  Falling back to local CSV → Iceberg table '{TABLE_NAME_RAW}'."
        )
        INPUT_SOURCE = "local_csv"
    else:
        raise FileNotFoundError(
            f"Missing Exercise 1 Parquet ({LOCAL_PARQUET}) and local CSV ({LOCAL_CSV}).\n"
            "Re-run 01_Basic_Deduplication.ipynb or ensure data/redundant_data.csv is in the project."
        )

if INPUT_SOURCE == "local_csv":
    INPUT_FORMAT = "csv"
    LOCAL_INPUT = LOCAL_CSV
    TABLE_NAME = TABLE_NAME_RAW
elif INPUT_SOURCE in ("exercise1_parquet", "parquet"):
    INPUT_FORMAT = "parquet"
    LOCAL_INPUT = LOCAL_PARQUET
    TABLE_NAME = TABLE_NAME_DEDUPED
else:
    raise ValueError(
        f"Unknown INPUT_SOURCE={INPUT_SOURCE!r}. Use 'exercise1_parquet' or 'local_csv'."
    )

# Optional override of the resolved table name
TABLE_NAME = os.environ.get("ICEBERG_TABLE", TABLE_NAME)
FULL_TABLE = f"{NAMESPACE}.{TABLE_NAME}"
INPUT_PATH = LOCAL_INPUT.resolve().as_uri() if LOCAL_INPUT.exists() else str(LOCAL_INPUT)

# --- Shared CDW / Hue (HiveServer2 JDBC via CML Data Connection) ---
# Connection snippet JDBC (reference — CML connection already embeds this):
# jdbc:hive2://hs2-default-hive-aws.dw-go01-demo-aws.ylcu-atmi.cloudera.site/default;
#   transportMode=http;httpPath=cliservice;socketTimeout=60;ssl=true;retries=3;
HIVE_JDBC_URL = os.environ.get(
    "HIVE_JDBC_URL",
    "jdbc:hive2://hs2-default-hive-aws.dw-go01-demo-aws.ylcu-atmi.cloudera.site/default;"
    "transportMode=http;httpPath=cliservice;socketTimeout=60;ssl=true;retries=3;",
)
# Publish into shared warehouse so Hue can see it (set False to skip Step 4b)
WRITE_TO_SHARED = os.environ.get("WRITE_TO_SHARED", "true").lower() in ("1", "true", "yes")
# Distinct Hue-facing name so it is obvious vs local-only lab tables
SHARED_TABLE_NAME = os.environ.get(
    "ICEBERG_SHARED_TABLE",
    f"{TABLE_NAME}_shared",
)
SHARED_FULL_TABLE = f"{NAMESPACE}.{SHARED_TABLE_NAME}"

# --- Optional: manual Iceberg REST catalog (cluster) ---
CATALOG_NAME = os.environ.get("ICEBERG_CATALOG_NAME", "iceberg")
REST_URI = os.environ.get("ICEBERG_REST_URI", "").strip()
REST_CREDENTIAL = os.environ.get("ICEBERG_REST_CREDENTIAL", "").strip()
REST_URI_CONFIGURED = bool(REST_URI) and "<" not in REST_URI and ">" not in REST_URI

# --- Local Iceberg Hadoop warehouse (session-only; NOT visible in Hue) ---
LOCAL_CATALOG = os.environ.get("ICEBERG_LOCAL_CATALOG", "local")
LOCAL_WAREHOUSE = Path(
    os.environ.get(
        "ICEBERG_LOCAL_WAREHOUSE",
        "/tmp/cdp_user_demo/phase1/iceberg-warehouse",
    )
).resolve()

print(f"CML connection: {CONNECTION_NAME}")
print(f"Input source:   {INPUT_SOURCE} ({INPUT_FORMAT})")
print(f"Input path:     {INPUT_PATH}")
print(f"Input exists:   {LOCAL_INPUT.exists()}")
print(f"Local table:    {FULL_TABLE} (session warehouse only)")
print(f"Shared/Hue:     {SHARED_FULL_TABLE} (WRITE_TO_SHARED={WRITE_TO_SHARED})")
print(f"Hive JDBC host: {HIVE_JDBC_URL.split('://')[1].split('/')[0] if '://' in HIVE_JDBC_URL else HIVE_JDBC_URL}")
print(f"Local warehouse:{LOCAL_WAREHOUSE}")
print(f"REST URI set:   {REST_URI_CONFIGURED}")


## Step 1b: Workload Credentials (Hive / Hue connection)

Hive JDBC connections need a **CDP workload password**. If `WORKLOAD_PASSWORD` is not set under User Settings → Environment Variables, enter it here and click **Save credentials** before Step 4b.

These values are passed to `cmldata.get_connection(..., {"USERNAME", "PASSWORD"})` and also exported as env vars for the CML Hive client.


In [ ]:
import os
import getpass

WORKLOAD_CREDS = {
    "USERNAME": os.environ.get("WORKLOAD_USER")
    or os.environ.get("KRB_USER")
    or os.environ.get("HADOOP_USER_NAME")
    or "",
    "PASSWORD": os.environ.get("WORKLOAD_PASSWORD") or "",
}

try:
    import ipywidgets as widgets
    from IPython.display import display

    _user_w = widgets.Text(
        value=WORKLOAD_CREDS["USERNAME"],
        description="Username:",
        placeholder="CDP workload user",
        style={"description_width": "100px"},
        layout=widgets.Layout(width="420px"),
    )
    _pass_w = widgets.Password(
        value=WORKLOAD_CREDS["PASSWORD"],
        description="Password:",
        placeholder="WORKLOAD_PASSWORD",
        style={"description_width": "100px"},
        layout=widgets.Layout(width="420px"),
    )
    _status = widgets.HTML(
        value=(
            "<i>✓ Password already present in env</i>"
            if WORKLOAD_CREDS["PASSWORD"]
            else "<i>Enter credentials, then click Save before Step 4b.</i>"
        )
    )
    _btn = widgets.Button(description="Save credentials", button_style="primary")

    def _save_creds(_=None):
        WORKLOAD_CREDS["USERNAME"] = (_user_w.value or "").strip()
        WORKLOAD_CREDS["PASSWORD"] = _pass_w.value or ""
        if WORKLOAD_CREDS["USERNAME"]:
            os.environ["WORKLOAD_USER"] = WORKLOAD_CREDS["USERNAME"]
        if WORKLOAD_CREDS["PASSWORD"]:
            os.environ["WORKLOAD_PASSWORD"] = WORKLOAD_CREDS["PASSWORD"]
        _status.value = (
            f"<b>✓ Saved</b> user=<code>{WORKLOAD_CREDS['USERNAME'] or '(empty)'}</code>; "
            f"password={'set' if WORKLOAD_CREDS['PASSWORD'] else 'NOT SET'}"
        )
        print(
            f"✓ Credentials saved (user={WORKLOAD_CREDS['USERNAME']!r}, "
            f"password={'set' if WORKLOAD_CREDS['PASSWORD'] else 'NOT SET'})"
        )

    _btn.on_click(_save_creds)
    display(widgets.VBox([
        widgets.HTML(f"<b>Connection:</b> <code>{CONNECTION_NAME}</code>"),
        _user_w,
        _pass_w,
        _btn,
        _status,
    ]))
    if WORKLOAD_CREDS["PASSWORD"]:
        _save_creds()
except ImportError:
    print("ipywidgets not available — using prompts instead")
    _u = input(f"Workload username [{WORKLOAD_CREDS['USERNAME']}]: ").strip()
    if _u:
        WORKLOAD_CREDS["USERNAME"] = _u
    if not WORKLOAD_CREDS["PASSWORD"]:
        WORKLOAD_CREDS["PASSWORD"] = getpass.getpass("Workload password: ")
    if WORKLOAD_CREDS["USERNAME"]:
        os.environ["WORKLOAD_USER"] = WORKLOAD_CREDS["USERNAME"]
    if WORKLOAD_CREDS["PASSWORD"]:
        os.environ["WORKLOAD_PASSWORD"] = WORKLOAD_CREDS["PASSWORD"]
    print(
        f"✓ Credentials set (user={WORKLOAD_CREDS['USERNAME']!r}, "
        f"password={'set' if WORKLOAD_CREDS['PASSWORD'] else 'NOT SET'})"
    )


def get_cml_connection(connection_name: str = None):
    """Hive/Spark Data Connection with optional workload USERNAME/PASSWORD."""
    import cml.data_v1 as cmldata

    name = connection_name or CONNECTION_NAME
    params = {}
    user = (WORKLOAD_CREDS.get("USERNAME") or os.environ.get("WORKLOAD_USER") or "").strip()
    password = WORKLOAD_CREDS.get("PASSWORD") or os.environ.get("WORKLOAD_PASSWORD") or ""
    if user:
        params["USERNAME"] = user
    if password:
        params["PASSWORD"] = password
        os.environ["WORKLOAD_PASSWORD"] = password
    if not password:
        raise KeyError(
            "No workload password set. Use the Step 1b form (Save credentials), "
            "or set WORKLOAD_PASSWORD under User Settings → Environment Variables."
        )
    return cmldata.get_connection(name, params)


## Step 2: Create Spark Session

1. Try `cml.data_v1.get_connection(...).get_spark_session()` (Spark Data Lake connection).
2. Else local Spark + **local Hadoop Iceberg warehouse** under `/tmp/.../iceberg-warehouse` (works even if `go01-obsr-de` is SQL-only / pandas).
3. Optional: attach Iceberg REST catalog when `ICEBERG_REST_URI` is a real URL — never with a `<placeholder>`.


In [ ]:
import os
import pyspark
from pyspark.sql import SparkSession

spark = None
conn = None
SPARK_MODE = None  # "cml_spark" | "local_rest" | "local_hadoop"

# 1) Preferred: CAI Data Connection Spark session (Iceberg + cluster Hadoop already wired)
try:
    import cml.data_v1 as cmldata

    conn = cmldata.get_connection(CONNECTION_NAME)
    get_spark = getattr(conn, "get_spark_session", None)
    if callable(get_spark):
        spark = get_spark()
        SPARK_MODE = "cml_spark"
        print(f"✓ Spark from CML connection: {CONNECTION_NAME}")
    else:
        # SQL-only connection (pandas/JDBC) — explore DBs, then use local Iceberg warehouse
        try:
            sample = conn.get_pandas_dataframe("show databases")
            print(f"✓ CML connection '{CONNECTION_NAME}' is SQL-only (pandas). Sample databases:")
            print(sample)
        except Exception as e:
            print(f"⚠ CML connection '{CONNECTION_NAME}' has no get_spark_session: {e}")
        print("→ Falling back to local Spark + local Iceberg Hadoop warehouse")
except ImportError:
    print("⚠ cml.data_v1 not available — falling back to local Spark")
except Exception as e:
    print(f"⚠ CML get_connection failed ({e}) — falling back to local Spark")

# 2) Local Spark + Iceberg catalog (REST if configured, else Hadoop warehouse on local disk)
if spark is None:
    for _k in ("HADOOP_CONF_DIR", "HADOOP_HOME", "HADOOP_HDFS_HOME"):
        if _k in os.environ:
            print(f"⚠ Unsetting {_k}={os.environ[_k]} for local Spark startup")
            os.environ.pop(_k)

    _spark_mm = ".".join(pyspark.__version__.split(".")[:2])
    ICEBERG_PACKAGE = os.environ.get(
        "ICEBERG_SPARK_PACKAGE",
        f"org.apache.iceberg:iceberg-spark-runtime-{_spark_mm}_2.12:1.6.1",
    )

    builder = (
        SparkSession.builder
        .master("local[*]")
        .appName("Exercise2_Iceberg")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.hadoop.hadoop.security.authentication", "simple")
        .config("spark.hadoop.hadoop.security.authorization", "false")
        .config("spark.jars.packages", ICEBERG_PACKAGE)
        .config(
            "spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
        )
    )

    if REST_URI_CONFIGURED:
        builder = (
            builder
            .config("spark.sql.defaultCatalog", CATALOG_NAME)
            .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
            .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
            .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", REST_URI)
            .config(f"spark.sql.catalog.{CATALOG_NAME}.default-namespace", NAMESPACE)
        )
        if REST_CREDENTIAL:
            builder = builder.config(
                f"spark.sql.catalog.{CATALOG_NAME}.credential", REST_CREDENTIAL
            )
        SPARK_MODE = "local_rest"
        globals()["FULL_TABLE"] = f"{CATALOG_NAME}.{NAMESPACE}.{TABLE_NAME}"
        print(f"✓ Local Spark + Iceberg REST catalog '{CATALOG_NAME}'")
        print(f"  REST URI: {REST_URI}")
    else:
        LOCAL_WAREHOUSE.mkdir(parents=True, exist_ok=True)
        builder = (
            builder
            .config("spark.sql.defaultCatalog", LOCAL_CATALOG)
            .config(f"spark.sql.catalog.{LOCAL_CATALOG}", "org.apache.iceberg.spark.SparkCatalog")
            .config(f"spark.sql.catalog.{LOCAL_CATALOG}.type", "hadoop")
            .config(f"spark.sql.catalog.{LOCAL_CATALOG}.warehouse", str(LOCAL_WAREHOUSE))
            .config(f"spark.sql.catalog.{LOCAL_CATALOG}.default-namespace", NAMESPACE)
        )
        SPARK_MODE = "local_hadoop"
        globals()["FULL_TABLE"] = f"{LOCAL_CATALOG}.{NAMESPACE}.{TABLE_NAME}"
        print(f"✓ Local Spark + Iceberg Hadoop catalog '{LOCAL_CATALOG}'")
        print(f"  Warehouse: {LOCAL_WAREHOUSE}")

    spark = builder.getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark master:  {spark.sparkContext.master}")
print(f"Mode:          {SPARK_MODE}")
print(f"Write target:  {FULL_TABLE}")


## Step 3: Load Source Data

Loads based on `INPUT_SOURCE` from Step 1:
- `exercise1_parquet` → Exercise 1 deduped Parquet (if missing under `/tmp`, Step 1 auto-falls back to CSV)
- `local_csv` → project `../data/redundant_data.csv` → table `raw_customers`


In [ ]:
assert LOCAL_INPUT.exists(), (
    f"Missing input for INPUT_SOURCE={INPUT_SOURCE!r}: {LOCAL_INPUT}\n"
    + (
        "Re-run 01_Basic_Deduplication.ipynb, or set EXERCISE1_PARQUET."
        if INPUT_FORMAT == "parquet"
        else "Ensure use-case-phase-1/data/redundant_data.csv is in the project, or set LOCAL_CSV."
    )
)

if INPUT_FORMAT == "csv":
    df = spark.read.csv(INPUT_PATH, header=True, inferSchema=True)
else:
    df = spark.read.parquet(INPUT_PATH)

print(f"✓ Loaded ({INPUT_FORMAT}): {INPUT_PATH}")
print(f"Will write Iceberg table: {FULL_TABLE}")
print(f"Rows: {df.count():,}")
print(f"Columns: {', '.join(df.columns)}")
df.show(10, truncate=False)
df.printSchema()


## Step 4: Create Namespace and Write Iceberg Table

Works with `cml_spark`, `local_rest`, or `local_hadoop` (session-local warehouse under `/tmp/.../iceberg-warehouse`).

Uses `DROP TABLE IF EXISTS` + `create()` so a half-written local table (missing `version-hint.text`) does not break the write. A one-time Hadoop `version-hint` WARN on first create is harmless if the cell still prints success.


In [ ]:
import shutil

assert SPARK_MODE in ("cml_spark", "local_rest", "local_hadoop"), (
    "Unexpected SPARK_MODE for Iceberg write: "
    f"{SPARK_MODE}. Re-run Step 2 after a kernel restart."
)

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {NAMESPACE}")
print(f"✓ Namespace ready: {NAMESPACE}")

# Prefer drop+create over createOrReplace: Hadoop catalog often WARNs/fails when
# metadata/ exists without version-hint.text (partial prior write under /tmp).
try:
    spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE}")
except Exception as e:
    print(f"⚠ DROP TABLE noted: {e}")

if SPARK_MODE == "local_hadoop":
    table_dir = LOCAL_WAREHOUSE / NAMESPACE / TABLE_NAME
    if table_dir.exists():
        shutil.rmtree(table_dir)
        print(f"✓ Cleared local table dir: {table_dir}")

(
    df.writeTo(FULL_TABLE)
    .using("iceberg")
    .tableProperty("write.format.default", "parquet")
    .create()
)

n = spark.table(FULL_TABLE).count()
print(f"✓ Iceberg table written: {FULL_TABLE} ({n:,} rows)")
if SPARK_MODE == "local_hadoop":
    meta = LOCAL_WAREHOUSE / NAMESPACE / TABLE_NAME / "metadata"
    hint = meta / "version-hint.text"
    print(f"  Warehouse: {LOCAL_WAREHOUSE}")
    print(f"  version-hint.text: {'present' if hint.is_file() else 'missing'}")


## Step 4b: Publish to Shared CDW Warehouse (visible in Hue)

Local Iceberg under `/tmp/.../iceberg-warehouse` is **only on this CAI session**. Hue queries the **shared Hive Metastore** behind your CDW Virtual Warehouse (`hs2-default-hive-aws...`).

This step uses the CML Hive JDBC connection (`go01-obsr-de`) to:
1. `CREATE DATABASE` / `CREATE TABLE ... USING ICEBERG` in the shared warehouse
2. `INSERT` rows from the loaded DataFrame
3. Verify with `SHOW TABLES` / `SELECT` through the same connection

Then open **Hue → Table Browser** (same environment / VW) and look for `SHARED_FULL_TABLE`.


In [ ]:
import math

if not WRITE_TO_SHARED:
    print("Skipped — WRITE_TO_SHARED is False")
else:
    import cml.data_v1 as cmldata

    hive_conn = cmldata.get_connection(CONNECTION_NAME)

    def hive_type(spark_dtype: str) -> str:
        d = spark_dtype.lower()
        if d.startswith("int"):
            return "BIGINT" if "64" in d or d == "long" else "INT"
        if d.startswith("bigint") or d == "long":
            return "BIGINT"
        if d.startswith("double") or d.startswith("float") or d.startswith("decimal"):
            return "DOUBLE"
        if d.startswith("boolean"):
            return "BOOLEAN"
        if d.startswith("timestamp"):
            return "TIMESTAMP"
        if d.startswith("date"):
            return "DATE"
        return "STRING"

    def sql_literal(value):
        if value is None:
            return "NULL"
        if isinstance(value, bool):
            return "TRUE" if value else "FALSE"
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
                return "NULL"
            return str(value)
        s = str(value).replace("\\", "\\\\").replace("'", "''")
        return f"'{s}'"

    col_defs = ", ".join(f"`{c}` {hive_type(t)}" for c, t in df.dtypes)
    create_candidates = [
        f"CREATE TABLE {SHARED_FULL_TABLE} ({col_defs}) USING ICEBERG",
        f"CREATE TABLE {SHARED_FULL_TABLE} ({col_defs}) STORED AS PARQUET",
    ]

    cursor = hive_conn.get_cursor()
    try:
        cursor.execute(f"CREATE DATABASE IF NOT EXISTS {NAMESPACE}")
        print(f"✓ Database ready: {NAMESPACE}")

        try:
            cursor.execute(f"DROP TABLE IF EXISTS {SHARED_FULL_TABLE}")
            print(f"✓ Dropped existing {SHARED_FULL_TABLE} (if any)")
        except Exception as e:
            print(f"⚠ DROP TABLE: {e}")

        created = False
        last_err = None
        for create_sql in create_candidates:
            try:
                cursor.execute(create_sql)
                print(f"✓ Created table: {SHARED_FULL_TABLE}")
                print(f"  DDL: {create_sql}")
                created = True
                break
            except Exception as e:
                last_err = e
                print(f"⚠ Create failed, trying next DDL style: {e}")
        if not created:
            raise RuntimeError(f"Could not create shared table: {last_err}")

        # Batch INSERT via Hive SQL (lab-scale data)
        cols = df.columns
        col_list = ", ".join(f"`{c}`" for c in cols)
        pdf = df.toPandas()
        batch_size = 100
        inserted = 0
        for start in range(0, len(pdf), batch_size):
            chunk = pdf.iloc[start : start + batch_size]
            values_sql = []
            for row in chunk.itertuples(index=False, name=None):
                values_sql.append("(" + ", ".join(sql_literal(v) for v in row) + ")")
            insert_sql = (
                f"INSERT INTO {SHARED_FULL_TABLE} ({col_list}) VALUES "
                + ", ".join(values_sql)
            )
            cursor.execute(insert_sql)
            inserted += len(chunk)
        print(f"✓ Inserted {inserted:,} rows into {SHARED_FULL_TABLE}")

        tables = hive_conn.get_pandas_dataframe(f"SHOW TABLES IN {NAMESPACE}")
        print("\n=== Tables in shared database (JDBC) ===")
        print(tables)

        preview = hive_conn.get_pandas_dataframe(
            f"SELECT * FROM {SHARED_FULL_TABLE} LIMIT 10"
        )
        print(f"\n=== Preview {SHARED_FULL_TABLE} ===")
        print(preview)
        print(
            f"\n→ In Hue: Table Browser → database `{NAMESPACE}` → "
            f"`{SHARED_TABLE_NAME}` (VW: default-hive-aws)."
        )
    finally:
        try:
            hive_conn.close()
        except Exception:
            pass


## Step 5: Find the Table in the Catalog

List namespaces/tables and confirm the new table is discoverable.


In [ ]:
print("=== Databases / namespaces ===")
spark.sql("SHOW NAMESPACES").show(truncate=False)

print(f"=== Tables in {NAMESPACE} ===")
tables_df = spark.sql(f"SHOW TABLES IN {NAMESPACE}")
tables_df.show(truncate=False)

rows = tables_df.collect()
table_names = []
for r in rows:
    d = r.asDict()
    table_names.append(d.get("tableName") or d.get("name") or d.get("table") or str(r[0]))

found = TABLE_NAME in table_names or any(TABLE_NAME == str(n) for n in table_names)
print(f"Looking for table: {TABLE_NAME}")
print(f"Tables found: {table_names}")
print("✓ Table found in catalog" if found else "✗ Table NOT found — check write step / namespace")


In [ ]:
print("=== DESCRIBE TABLE ===")
spark.sql(f"DESCRIBE TABLE EXTENDED {FULL_TABLE}").show(100, truncate=False)

print("=== Sample query from Iceberg table ===")
iceberg_df = spark.table(FULL_TABLE)
print(f"Rows in Iceberg table: {iceberg_df.count():,}")
iceberg_df.show(10, truncate=False)


## Step 6 (Optional): Query the REST Catalog HTTP API Directly

Only when `ICEBERG_REST_URI` is configured. Skip for CML Spark mode — use Step 5 Spark SQL instead.


In [ ]:
import json
import urllib.request
import urllib.error
import base64

if not REST_URI_CONFIGURED:
    print("Skipped — ICEBERG_REST_URI not set. Use Step 5 Spark SQL for catalog discovery.")
else:
    def rest_get(path: str):
        """GET a path under the Iceberg REST catalog URI."""
        url = REST_URI.rstrip("/") + path
        req = urllib.request.Request(url, method="GET")
        req.add_header("Accept", "application/json")

        token = os.environ.get("ICEBERG_REST_TOKEN", "")
        if token:
            req.add_header("Authorization", f"Bearer {token}")
        elif REST_CREDENTIAL:
            encoded = base64.b64encode(REST_CREDENTIAL.encode("utf-8")).decode("ascii")
            req.add_header("Authorization", f"Basic {encoded}")

        with urllib.request.urlopen(req, timeout=30) as resp:
            return json.loads(resp.read().decode("utf-8"))

    try:
        ns_path = NAMESPACE.replace(".", "%1F")
        payload = rest_get(f"/v1/namespaces/{ns_path}/tables")
        identifiers = payload.get("identifiers", [])
        print("REST API tables in namespace:")
        print(json.dumps(identifiers, indent=2))

        matched = [
            i for i in identifiers
            if i.get("name") == TABLE_NAME or TABLE_NAME in str(i)
        ]
        if matched:
            print(f"\n✓ Found via REST API: {matched}")
            meta = rest_get(f"/v1/namespaces/{ns_path}/tables/{TABLE_NAME}")
            print("\nTable metadata keys:", list(meta.keys()))
        else:
            print(f"\n✗ Table '{TABLE_NAME}' not returned by REST list endpoint")
    except urllib.error.HTTPError as e:
        print(f"REST API HTTP error: {e.code} {e.reason}")
        print("Spark SQL discovery in Step 5 may still succeed.")
    except Exception as e:
        print(f"REST API call skipped/failed: {e}")


## Summary

| Item | Value |
|------|-------|
| Input A | Exercise 1 Parquet → `deduped_customers` |
| Input B | Local CSV → `raw_customers` |
| Local only | Hadoop Iceberg under `/tmp/.../iceberg-warehouse` (**not** in Hue) |
| Shared / Hue | `cdp_user_demo.<table>_shared` via Hive JDBC (`go01-obsr-de` / CDW HS2) |
| JDBC | `jdbc:hive2://hs2-default-hive-aws.dw-go01-demo-aws.ylcu-atmi.cloudera.site/default;...` |

### Key Takeaways

- **Hue only sees shared Metastore / CDW tables**, not CAI session `/tmp` warehouses
- Use the Hive VW Data Connection (`get_cursor` / `get_pandas_dataframe`) to publish lab data for Hue
- Prefer Iceberg (`USING ICEBERG`); notebook falls back to `STORED AS PARQUET` if needed
- Keep distinct names for local vs shared (`*_shared`) so sources do not collide

## Cleanup


In [ ]:
spark.stop()
if conn is not None and hasattr(conn, "close"):
    try:
        conn.close()
    except Exception:
        pass
print("✓ Spark session stopped")
